In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Транспіляція за допомогою менеджерів проходів

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Рекомендований спосіб транспіляції схеми — створити поетапний менеджер проходів і виконати його метод `run`, передавши схему як вхідні дані. На цій сторінці пояснюється, як транспілювати квантові схеми таким чином.
## Що таке (поетапний) менеджер проходів?
У контексті Qiskit SDK транспіляція означає процес перетворення вхідної схеми у форму, придатну для виконання на квантовому пристрої. Транспіляція, як правило, відбувається у послідовності кроків, що називаються проходами транспілятора. Схема обробляється кожним проходом транспілятора послідовно, при цьому вихід одного проходу стає входом наступного. Наприклад, один прохід може пройтися по схемі та об'єднати всі послідовні однокубітні вентилі, а наступний прохід може синтезувати ці вентилі у базовий набір цільового пристрою. Проходи транспілятора, що входять до складу Qiskit, розташовані в модулі [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Менеджер проходів — це об'єкт, що зберігає список проходів транспілятора і може виконувати їх на схемі. Щоб створити менеджер проходів, ініціалізуй [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) зі списком проходів транспілятора. Щоб запустити транспіляцію схеми, виклич метод [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run), передавши схему як вхідні дані.

Поетапний менеджер проходів — це особливий вид менеджера проходів, що представляє рівень абстракції вище, ніж звичайний менеджер проходів. Тоді як звичайний менеджер проходів складається з кількох проходів транспілятора, поетапний менеджер проходів складається з кількох *менеджерів проходів*. Це корисна абстракція, оскільки транспіляція, як правило, відбувається в окремих етапах, як описано в розділі [Етапи транспілятора](transpiler-stages), де кожен етап представлений менеджером проходів. Поетапні менеджери проходів представлені класом [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). Решта цієї сторінки описує, як створювати та налаштовувати (поетапні) менеджери проходів.
## Генерація наперед визначеного поетапного менеджера проходів
Щоб створити наперед визначений поетапний менеджер проходів з розумними стандартними параметрами, використовуй функцію [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Щоб транспілювати схему або список схем за допомогою менеджера проходів, передай схему або список схем у метод `run`. Зробімо це для двокубітної схеми, що складається з вентиля Адамара, за яким ідуть два суміжні вентилі CX:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Перегляньте [Стандартні налаштування та параметри конфігурації транспіляції](defaults-and-configuration-options) для опису можливих аргументів функції `generate_preset_pass_manager`. Аргументи `generate_preset_pass_manager` збігаються з аргументами функції [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Having trouble remembering pass manager details? Try asking Qiskit Code Assistant." />

Якщо наперед визначені менеджери проходів не задовольняють твої потреби, налаштуй транспіляцію, створивши (поетапні) менеджери проходів або навіть проходи транспіляції. Решта цієї сторінки описує, як створювати менеджери проходів. Інструкції зі створення проходів транспіляції дивись у розділі [Написання власного проходу транспілятора](custom-transpiler-pass).

## Створення власного менеджера проходів

Модуль [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) містить багато проходів транспілятора, що можуть використовуватися для створення менеджерів проходів. Щоб створити менеджер проходів, ініціалізуй `PassManager` зі списком проходів. Наприклад, наступний код створює прохід транспілятора, що об'єднує суміжні двокубітні вентилі, а потім синтезує їх у базис [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) та [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Щоб продемонструвати цей менеджер проходів у дії, протестуй його на двокубітній схемі, що складається з вентиля Адамара, за яким ідуть два суміжні вентилі CX:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Щоб запустити менеджер проходів на схемі, виклич метод `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Більш просунутий приклад, що показує, як створити менеджер проходів для реалізації техніки придушення помилок, відомої як динамічне розв'язання, дивись у розділі [Створення менеджера проходів для динамічного розв'язання](dynamical-decoupling-pass-manager).

## Створення поетапного менеджера проходів

`StagedPassManager` — це менеджер проходів, що складається з окремих етапів, де кожен етап визначається екземпляром `PassManager`. Ти можеш створити `StagedPassManager`, задавши бажані етапи. Наприклад, наступний код створює поетапний менеджер проходів з двома етапами, `init` та `translation`. Етап `translation` визначається менеджером проходів, створеним раніше.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Кількість етапів у поетапному менеджері проходів не обмежена.

Ще один корисний спосіб створення поетапного менеджера проходів — почати з наперед визначеного поетапного менеджера проходів і замінити деякі з його етапів. Наприклад, наступний код генерує наперед визначений менеджер проходів з рівнем оптимізації 3, а потім задає кастомний етап `pre_layout`.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[Функції-генератори етапів](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) можуть бути корисними для побудови кастомних менеджерів проходів.
Вони генерують етапи, що забезпечують загальну функціональність, яка використовується в багатьох менеджерах проходів.
Наприклад, [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) може використовуватися для генерації етапу
"вбудовування" вибраного початкового `Layout` з проходу розмітки на вказаний цільовий пристрій.

## Наступні кроки
> **Tip:** - [Написати кастомний прохід транспілятора](custom-transpiler-pass).
>     - [Створити менеджер проходів для динамічного розв'язання](dynamical-decoupling-pass-manager).
>     - Щоб дізнатися більше про функцію `generate_preset_passmanager`, перегляньте розділ [Стандартні налаштування транспіляції та параметри конфігурації](defaults-and-configuration-options).
>     - Спробуй посібник [Порівняння налаштувань транспілятора](/guides/circuit-transpilation-settings).
>     - Перегляньте [документацію API транспілятора](https://docs.quantum.ibm.com/api/qiskit/transpiler).